# Modelado — Cancelaciones de reservas hoteleras

- **Proyecto:** Práctica Final · Machine Learning y Deep Learning
- **Máster:** Inteligencia Artificial, Cloud Computing y DevOps · PontIA.tech
- **Fase:** 4 — Modelado

## Objetivo

Entrenar y evaluar 5 modelos de clasificación binaria para predecir cancelaciones:

1. LogisticRegression (baseline)
2. DecisionTreeClassifier
3. RandomForestClassifier
4. XGBClassifier (Gradient Boosting)
5. Red Neuronal con Keras

Todos los experimentos se registran en MLflow para comparación sistemática.

## Métrica principal

ROC-AUC (siguiendo la decisión tomada en Sección 2 del EDA).
Métrica secundaria: F1-score.

In [17]:
"""Setup del notebook de modelado."""

import sys
from pathlib import Path

# Path setup
PATH_PROYECTO = Path.cwd().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PATH_PROYECTO))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports de nuestros módulos
from src.data_loader import preparar_datos, SEED
from src.models import (
    configurar_mlflow,
    calcular_metricas,
    imprimir_metricas,
)

print(f"Raíz del proyecto: {PATH_PROYECTO}")
print(f"SEED: {SEED}")

Raíz del proyecto: c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml
SEED: 42


## Preparación de datos

Reusamos `preparar_datos()` del módulo `data_loader.py` (Fase 3). Una sola llamada
hace todo el preprocesamiento: carga, limpieza, encoding, escalado, split.

In [18]:
"""Cargar datos preprocesados."""

X_train, X_test, y_train, y_test, preprocesador = preparar_datos()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape} (cancela: {y_train.mean()*100:.2f}%)")
print(f"y_test:  {y_test.shape} (cancela: {y_test.mean()*100:.2f}%)")

Eliminadas 13 filas con valores imposibles (0.011% del total).
X_train: (95501, 130)
X_test:  (23876, 130)
y_train: (95501,) (cancela: 37.04%)
y_test:  (23876,) (cancela: 37.04%)


## Setup de MLflow

MLflow es la herramienta de tracking de experimentos. Cada vez que entrenamos
un modelo, MLflow registra automáticamente:
- Hiperparámetros usados.
- Métricas obtenidas.
- El modelo entrenado (para poder recuperarlo después).

Los runs se guardan en la carpeta `mlruns/` del proyecto (ya añadida a `.gitignore`
en el siguiente paso para no commitearla).

In [19]:
"""Configurar MLflow."""

configurar_mlflow()

MLflow configurado.
  Tracking URI: file:///C:/Juan/Pontia/ML/practica-final-ml/practica-final-ml/mlruns
  Experimento: cancelaciones_hoteleras


## Modelo 1 — LogisticRegression (baseline)

Empezamos con el modelo más simple: regresión logística sin balanceo de clases.
Es nuestro baseline contra el que compararemos el resto de modelos.

**Por qué es el baseline**:

- Es un modelo lineal: aprende una combinación lineal de las features.
- Es rápido de entrenar (segundos en este dataset).
- Es interpretable: podemos ver qué peso aprende para cada variable.
- Cualquier modelo más complejo debe superarlo claramente para justificar
  su complejidad añadida.

**Limitación**:

- Solo capta relaciones LINEALES. Si la relación entre features y target
  es no lineal (interacciones complejas, umbrales, efectos no monótonos),
  no la detectará. Los modelos posteriores (RandomForest, XGBoost) sí.

**Hipótesis a priori**: ROC-AUC entre 0.82 y 0.86. Bueno pero mejorable.

In [20]:
"""Entrenar LogisticRegression (baseline)."""

import importlib
from src import models
importlib.reload(models)

from src.models import entrenar_logistic_regression, imprimir_metricas

modelo_lr, metricas_lr = entrenar_logistic_regression(
    X_train, y_train, X_test, y_test
)

imprimir_metricas(metricas_lr, "LogisticRegression (baseline)")

2026/05/28 13:35:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/28 13:35:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



METRICAS: LogisticRegression (baseline)
  accuracy    : 0.8313
  precision   : 0.8110
  recall      : 0.7098
  f1          : 0.7570
  roc_auc     : 0.9102


## Modelo 2 — DecisionTreeClassifier

Segundo modelo: árbol de decisión. A diferencia de LogisticRegression, capta
relaciones no lineales y umbrales mediante una cascada de preguntas binarias
sobre las features.

**Hiperparámetros usados**:

- `max_depth=10`: limita la profundidad del árbol para evitar overfitting.
- `min_samples_split=20`: requiere al menos 20 muestras para dividir un nodo.
- `min_samples_leaf=10`: requiere al menos 10 muestras en cada hoja.

**Hipótesis a priori**: ROC-AUC entre 0.85 y 0.91. Probablemente mejor que
LogisticRegression por captar no linealidad, pero peor que los ensembles
(RandomForest, XGBoost) que vienen después.

In [21]:
"""Entrenar DecisionTreeClassifier."""

import importlib
from src import models
importlib.reload(models)

from src.models import entrenar_decision_tree, imprimir_metricas

modelo_dt, metricas_dt = entrenar_decision_tree(
    X_train, y_train, X_test, y_test
)

imprimir_metricas(metricas_dt, "DecisionTree (max_depth=10)")

2026/05/28 13:35:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/28 13:35:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



METRICAS: DecisionTree (max_depth=10)
  accuracy    : 0.8511
  precision   : 0.8423
  recall      : 0.7357
  f1          : 0.7854
  roc_auc     : 0.9285


## Modelo 3 — RandomForestClassifier

Tercer modelo: Random Forest, un ensemble de 100 árboles de decisión.

A diferencia de un árbol solo (que tiende al overfitting), el Random Forest
entrena muchos árboles sobre muestras aleatorias de filas y features, y
promedia sus predicciones. Esto reduce el ruido individual y mejora la
generalización.

**Hiperparámetros usados**:

- `n_estimators=100`: número de árboles en el bosque.
- `max_depth=20`: profundidad máxima de cada árbol.
- `min_samples_split=10`, `min_samples_leaf=5`: controlan el crecimiento.
- `n_jobs=-1`: usa todos los cores del procesador para paralelizar.

**Hipótesis a priori**: ROC-AUC entre 0.93 y 0.96. Debería superar al
DecisionTree (0.9285) por ser un ensemble más robusto.

**Nota**: entrena 100 árboles, así que tarda más (1-3 minutos).

In [22]:
"""Entrenar RandomForestClassifier."""

import importlib
from src import models
importlib.reload(models)

from src.models import entrenar_random_forest, imprimir_metricas

modelo_rf, metricas_rf = entrenar_random_forest(
    X_train, y_train, X_test, y_test
)

imprimir_metricas(metricas_rf, "RandomForest (100 arboles)")

2026/05/28 13:35:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/28 13:35:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



METRICAS: RandomForest (100 arboles)
  accuracy    : 0.8703
  precision   : 0.8911
  recall      : 0.7404
  f1          : 0.8088
  roc_auc     : 0.9485


## Modelo 4 — XGBoost (Gradient Boosting)

Cuarto modelo: XGBoost, el algoritmo de referencia para datos tabulares.

A diferencia de Random Forest (árboles paralelos e independientes), XGBoost
entrena árboles de forma SECUENCIAL: cada árbol nuevo corrige los errores
del anterior. Esta corrección iterativa (gradient boosting) suele dar el
mejor rendimiento en datos tabulares.

**Hiperparámetros usados**:

- `n_estimators=300`: número de árboles (rondas de boosting).
- `max_depth=6`: profundidad de cada árbol (más bajo que RF porque el
  boosting combina muchos árboles débiles).
- `learning_rate=0.1`: cuánto contribuye cada árbol. Bajo = más robusto.
- `subsample=0.8`: cada árbol usa el 80% de las filas (reduce overfitting).
- `colsample_bytree=0.8`: cada árbol usa el 80% de las features.

**Hipótesis a priori**: ROC-AUC entre 0.94 y 0.97. Probablemente el mejor
modelo del proyecto.

In [23]:
"""Entrenar XGBoost."""

import importlib
from src import models
importlib.reload(models)

from src.models import entrenar_xgboost, imprimir_metricas

modelo_xgb, metricas_xgb = entrenar_xgboost(
    X_train, y_train, X_test, y_test
)

imprimir_metricas(metricas_xgb, "XGBoost (300 arboles)")

2026/05/28 13:35:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/28 13:35:17 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



METRICAS: XGBoost (300 arboles)
  accuracy    : 0.8798
  precision   : 0.8574
  recall      : 0.8104
  f1          : 0.8332
  roc_auc     : 0.9519


## Modelo 5 — Red Neuronal con Keras

Quinto y último modelo base: una red neuronal feed-forward construida con Keras.

A diferencia de los modelos basados en árboles, la red neuronal aprende
ajustando los pesos de sus neuronas mediante descenso de gradiente, a lo
largo de varias épocas de entrenamiento. Necesita las features escaladas
(que ya tenemos del preprocesamiento).

**Arquitectura**:

- Capa de entrada: 130 features.
- 3 capas ocultas: 128 → 64 → 32 neuronas, activación ReLU.
- Dropout del 30% entre capas (apaga neuronas aleatoriamente para evitar overfitting).
- Capa de salida: 1 neurona con sigmoide (probabilidad de cancelación).

**Hiperparámetros de entrenamiento**:

- Optimizador Adam con learning_rate=0.001.
- Loss: binary_crossentropy (estándar para clasificación binaria).
- Hasta 50 épocas, pero con EarlyStopping (para si no mejora en 5 épocas).
- Batch size de 256.

**Hipótesis a priori**: ROC-AUC entre 0.92 y 0.95. En datos tabulares, las
redes raramente superan a XGBoost, pero aportan diversidad al conjunto.

In [24]:
"""Entrenar Red Neuronal con Keras."""

import importlib
from src import models
importlib.reload(models)

from src.models import entrenar_red_neuronal, imprimir_metricas

modelo_nn, metricas_nn = entrenar_red_neuronal(
    X_train, y_train, X_test, y_test
)

imprimir_metricas(metricas_nn, "Red Neuronal (128-64-32)")

ImportError: cannot import name 'entrenar_red_neuronal' from 'src.models' (c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml\src\models.py)

## Sub-fase 4.7 — Optimización de hiperparámetros

Hasta ahora hemos usado hiperparámetros elegidos a mano con criterio. Ahora
los optimizamos sistemáticamente con **RandomizedSearchCV**, que prueba muchas
combinaciones aleatorias y se queda con la mejor según ROC-AUC (con validación
cruzada).

La lógica de optimización vive en un módulo separado, `src/optimization.py`,
distinto de `models.py`. Razón: responsabilidad única. `models.py` entrena con
hiperparámetros dados; `optimization.py` busca los mejores.

### Optimización 1 — XGBoost (modelo ganador)

Espacio de búsqueda con 6 hiperparámetros. Con `n_iter=20` combinaciones y
`cv=3` folds = 60 entrenamientos.

**Objetivo**: intentar superar el ROC-AUC base de 0.9519.

In [ ]:
"""Optimizar XGBoost con RandomizedSearchCV."""

import importlib
from src import optimization
importlib.reload(optimization)

from src.optimization import optimizar_xgboost

modelo_xgb_opt, metricas_xgb_opt, params_xgb_opt = optimizar_xgboost(
    X_train, y_train, X_test, y_test,
    n_iter=20, cv=3
)

print("\nMejores hiperparametros encontrados:")
for nombre, valor in params_xgb_opt.items():
    print(f"  {nombre}: {valor}")

from src.models import imprimir_metricas
imprimir_metricas(metricas_xgb_opt, "XGBoost OPTIMIZADO")

Fitting 3 folds for each of 20 candidates, totalling 60 fits


2026/05/28 11:25:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/28 11:25:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Mejores hiperparametros encontrados:
  subsample: 1.0
  n_estimators: 500
  min_child_weight: 3
  max_depth: 7
  learning_rate: 0.2
  colsample_bytree: 0.8

METRICAS: XGBoost OPTIMIZADO
  accuracy    : 0.8857
  precision   : 0.8617
  recall      : 0.8237
  f1          : 0.8423
  roc_auc     : 0.9577


### Optimización 2 — RandomForest

Aplicamos el mismo proceso (RandomizedSearchCV) al RandomForest, segundo
mejor modelo base (ROC-AUC 0.9485).

Espacio de búsqueda con 5 hiperparámetros: n_estimators, max_depth,
min_samples_split, min_samples_leaf, max_features.

**Objetivo**: intentar superar el ROC-AUC base de 0.9485.

In [ ]:
"""Optimizar RandomForest con RandomizedSearchCV."""

from src.optimization import optimizar_random_forest

modelo_rf_opt, metricas_rf_opt, params_rf_opt = optimizar_random_forest(
    X_train, y_train, X_test, y_test,
    n_iter=20, cv=3
)

print("\nMejores hiperparametros encontrados:")
for nombre, valor in params_rf_opt.items():
    print(f"  {nombre}: {valor}")

imprimir_metricas(metricas_rf_opt, "RandomForest OPTIMIZADO")

ModuleNotFoundError: No module named 'src'